# 10 — Replication in an independent study (Kimmel et al. 2019)

Every check so far lives inside one atlas, where age and processing batch move together. **Kimmel et al. 2019** (Genome Research) profiled young and old mouse lung, kidney and spleen with the same 10x technology in another lab, with other mice. A model trained on Tabula Muris Senis and tested there **cannot be reading TMS's batch**, so a cell type that transfers has an age signal this atlas design can't fake.

Design facts from the paper: male C57BL/6J only; young 7–8 months (29–34 weeks), old about 20–23 months (the paper gives both 88–93 weeks and 22–23 months); 4 young and 3 old mice (lung: 3 and 3, one young lung removed by the authors for a batch effect); all killed at the same time of day; genes annotated with GENCODE vM20. The paper does not say whether young and old mice shared processing days.

**What is tested is direction, not age.** TMS models are trained on 1–3 months vs 18–30 months; Kimmel's young mice are 7–8 months, between the two. A model that learned aging should still rank Kimmel's old mice above its young ones. AUROC is computed within Kimmel, so a constant platform offset per gene cannot create a result.

### 1. Setup

In [ ]:
import sys
sys.path.append("..")          # so "src" can be imported from the notebook folder

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from sklearn.linear_model import Ridge
from sklearn.metrics import roc_auc_score
from src import aging as ag

A = ag.paths("atlas")
atlas = pd.read_csv(A.figure("atlas.csv"))
atlas["robust"] = atlas["verdict"].eq("age signal") & ~atlas["tech_confounded"].astype(bool)
KIM = ag.ROOT / "data" / "external" / "Kimmel_GR_2019_data"

### 2. Load Kimmel and normalize it the same way

Kimmel's `.X` holds raw integer counts. Normalized like TMS in 03: counts per 10,000, then log(1 + x). Mouse = age + animal number.

In [ ]:
kim = {}
for t in ["lung", "kidney", "spleen"]:
    k = sc.read_h5ad(KIM / f"{t}.h5ad")
    k.layers["counts"] = k.X.copy()
    sc.pp.normalize_total(k, target_sum=1e4)
    sc.pp.log1p(k)
    k.obs["mouse"] = k.obs["age"].astype(str) + "-" + k.obs["animal"].astype(str)
    kim[t] = k
    print(t, k.shape, k.obs.groupby("age", observed=True)["mouse"].nunique().to_dict())

### 3. Match cell types

Each TMS row is matched to a Kimmel label (`cell_type`, or `subtype` where Kimmel is coarser). Lung stromal subtypes are matched by their marker: *Npnt*+ stromal cells are alveolar fibroblasts (TMS "fibroblast of lung"), *Dcn*+ are adventitial fibroblasts. Only rows with a defensible match are listed; a TMS row without one is not tested.

In [ ]:
MATCH = [  # (TMS tissue, TMS cell type, Kimmel tissue, Kimmel column, Kimmel label)
    ("Lung", "fibroblast of lung", "lung", "subtype", "Npnt stromal cell"),
    ("Lung", "adventitial cell", "lung", "subtype", "Dcn stromal cell"),
    ("Lung", "type II pneumocyte", "lung", "cell_type", "type II pneumocyte"),
    ("Lung", "alveolar macrophage", "lung", "cell_type", "alveolar macrophage"),
    ("Lung", "classical monocyte", "lung", "cell_type", "classical monocyte"),
    ("Lung", "non-classical monocyte", "lung", "cell_type", "non-classical monocyte"),
    ("Lung", "B cell", "lung", "cell_type", "B cell"),
    ("Lung", "NK cell", "lung", "cell_type", "natural killer cell"),
    ("Lung", "CD4-positive, alpha-beta T cell", "lung", "subtype", "CD4 T cell"),
    ("Lung", "CD8-positive, alpha-beta T cell", "lung", "subtype", "CD8 T cell"),
    ("Kidney", "macrophage", "kidney", "cell_type", "macrophage"),
    ("Kidney", "kidney mesangial cell", "kidney", "cell_type", "mesangial cell"),
    ("Kidney", "fenestrated cell", "kidney", "cell_type", "kidney capillary endothelial cell"),
    ("Kidney", "kidney proximal convoluted tubule epithelial cell", "kidney", "cell_type",
     "kidney proximal straight tubule epithelial cell"),
    ("Kidney", "kidney collecting duct principal cell", "kidney", "cell_type", "kidney collecting duct epithelial cell"),
    ("Spleen", "B cell", "spleen", "cell_type", "B cell"),
    ("Spleen", "T cell", "spleen", "cell_type", "T cell"),
    ("Spleen", "NK cell", "spleen", "cell_type", "natural killer cell"),
    ("Spleen", "macrophage", "spleen", "cell_type", "macrophage"),
]
match = pd.DataFrame(MATCH, columns=["tissue", "cell_type", "kim_tissue", "kim_col", "kim_label"])
match = match.merge(atlas[["tissue", "cell_type", "compartment", "verdict", "robust", "cross_age_all_splits"]],
                    on=["tissue", "cell_type"], how="left")
match["kim_cells"] = [int((kim[r.kim_tissue].obs[r.kim_col] == r.kim_label).sum()) for r in match.itertuples()]
match

### 4. Train on TMS, test on Kimmel

Per row: logistic regression (the same model and `C` as 05a) trained on **all** TMS mice of the row (with its reviewed exclusions, 300 cells per mouse), on TMS's variable genes minus ambient, technical and sex genes (`feature_mask(..., "bio")`; Kimmel is all male), **restricted to genes Kimmel also has**. Then every Kimmel cell of the matched type is scored. Reported: AUROC over cells, AUROC over mice (mean score per mouse; with 6–7 mice only a few values are possible), and each Kimmel mouse's mean score.

A ridge clock (05c, adult TMS mice, `alpha=100`) is also applied: does it put Kimmel's old mice older than its young ones?

In [ ]:
results, mouse_scores = [], []
loaded = {}
for r in match.itertuples():
    if r.tissue not in loaded:
        loaded = {r.tissue: sc.read_h5ad(ag.paths(r.tissue).processed)}
    a = loaded[r.tissue]
    k = kim[r.kim_tissue]
    feat = ag.feature_mask(a, "bio") & a.var_names.isin(k.var_names)
    genes = a.var_names[feat]
    kw = ag.main_kw(r.cell_type, r.tissue)
    X, y, _ = ag.prep(a, r.cell_type, features=feat, **kw)
    clf = ag.logreg().fit(X, y)
    Xa, ya, _ = ag.prep(a, r.cell_type, features=feat, ages=ag.ADULT, target="months", **kw)
    clock = Ridge(alpha=100).fit(Xa, ya)

    kk = k[(k.obs[r.kim_col] == r.kim_label).to_numpy()]
    Xk = kk[:, genes].X
    Xk = Xk.toarray() if hasattr(Xk, "toarray") else np.asarray(Xk)
    yk = kk.obs["age"].eq("old").to_numpy().astype(int)
    p = clf.predict_proba(Xk)[:, 1]
    months = clock.predict(Xk)
    pm = pd.DataFrame({"mouse": kk.obs["mouse"].to_numpy(), "old": yk, "score": p, "pred_months": months}) \
           .groupby("mouse").agg(old=("old", "first"), score=("score", "mean"),
                                 pred_months=("pred_months", "median"), cells=("score", "size"))
    pm = pm[pm["cells"] >= 10]
    results.append({"tissue": r.tissue, "cell_type": r.cell_type, "genes": len(genes),
                    "kim_mice": len(pm),
                    "transfer_auroc_cell": roc_auc_score(yk, p) if len(set(yk)) == 2 else np.nan,
                    "transfer_auroc_mouse": roc_auc_score(pm["old"], pm["score"]) if pm["old"].nunique() == 2 else np.nan,
                    "clock_old_minus_young": pm.loc[pm["old"] == 1, "pred_months"].median()
                                             - pm.loc[pm["old"] == 0, "pred_months"].median()})
    mouse_scores.append(pm.reset_index().assign(tissue=r.tissue, cell_type=r.cell_type))
rep = match.merge(pd.DataFrame(results), on=["tissue", "cell_type"])
rep.round(3).to_csv(A.figure("replication_kimmel.csv"), index=False)
pd.concat(mouse_scores).round(3).to_csv(A.figure("replication_kimmel_by_mouse.csv"), index=False)
rep[["tissue", "cell_type", "compartment", "robust", "cross_age_all_splits", "kim_cells", "kim_mice",
     "transfer_auroc_cell", "transfer_auroc_mouse", "clock_old_minus_young"]].round(2) \
   .sort_values("transfer_auroc_cell", ascending=False)

### 5. Does the batch check predict transfer?

The test of the whole framework: rows that passed the TMS batch check should transfer to another lab better than rows that didn't. A positive correlation between `cross_age_all_splits` (inside TMS) and `transfer_auroc_cell` (in Kimmel) means the batch check measures something real.

In [ ]:
from scipy.stats import spearmanr
d = rep.dropna(subset=["transfer_auroc_cell", "cross_age_all_splits"])
rho, p = spearmanr(d["cross_age_all_splits"], d["transfer_auroc_cell"])
print(f"cross-age (TMS) vs transfer AUROC (Kimmel): Spearman rho = {rho:.2f}, p = {p:.3f}, n = {len(d)} rows")
print(d.groupby("robust")["transfer_auroc_cell"].agg(["size", "median", "min", "max"]).round(2))
if d["robust"].nunique() == 2:
    print("Mann-Whitney robust > other:", round(mannwhitneyu(d.loc[d.robust, "transfer_auroc_cell"],
                                                             d.loc[~d.robust, "transfer_auroc_cell"],
                                                             alternative="greater").pvalue, 4))

fig, ax = plt.subplots(figsize=(5, 4.5))
sns.scatterplot(data=d, x="cross_age_all_splits", y="transfer_auroc_cell", hue="compartment",
                style="tissue", s=70, ax=ax)
for r in d.itertuples():
    ax.annotate(r.cell_type[:18], (r.cross_age_all_splits, r.transfer_auroc_cell), fontsize=6,
                xytext=(3, 2), textcoords="offset points")
ax.axhline(0.5, color="gray", lw=0.6); ax.axvline(0.7, color="gray", lw=0.6, ls="--")
ax.set_xlabel("cross-age AUROC inside TMS (batch check)")
ax.set_ylabel("AUROC in Kimmel 2019 (another lab)")
ax.legend(fontsize=6, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.savefig(A.figure("replication_kimmel.png"), dpi=200, bbox_inches="tight")

### 5b. Could the transfer itself be technical?

The transfer AUROCs are weak (0.51–0.76), and a weak signal is exactly what a shared technical difference would give: if old cells yield less RNA (or more stress-response RNA) in **both** labs, a model can learn "lower depth = old" in TMS and find the same in Kimmel without any aging biology. Test: the technical-only model of 07 (log counts, log genes, ribosomal, dissociation and hemoglobin fractions per cell, each standardized within its own dataset and tissue) is trained on the TMS row and applied to the matched Kimmel cells, exactly like the expression model. Expression transfer that clearly beats technical transfer is not explained by these five features. Note also that the 19 rows are not 19 independent replications: they come from 3 tissues of the same 6–7 Kimmel mice.

In [ ]:
kim_tech = {t: ag.technical_adata(k) for t, k in kim.items()}
tech_rows, tms_tech = [], {}
for r in match.itertuples():
    if r.tissue not in tms_tech:
        tms_tech = {r.tissue: ag.technical_adata(sc.read_h5ad(ag.paths(r.tissue).processed))}
    X, y, _ = ag.prep(tms_tech[r.tissue], r.cell_type, **ag.main_kw(r.cell_type, r.tissue))
    clf = ag.logreg().fit(X, y)
    kt = kim_tech[r.kim_tissue]
    kk = kt[(kt.obs[r.kim_col] == r.kim_label).to_numpy()]
    yk = kk.obs["age"].eq("old").to_numpy().astype(int)
    p = clf.predict_proba(np.asarray(kk.X))[:, 1]
    pm = pd.DataFrame({"mouse": kk.obs["mouse"].to_numpy(), "old": yk, "score": p}) \
           .groupby("mouse").agg(old=("old", "first"), score=("score", "mean"))
    tech_rows.append({"tissue": r.tissue, "cell_type": r.cell_type,
                      "tech_transfer_auroc_cell": roc_auc_score(yk, p),
                      "tech_transfer_auroc_mouse": roc_auc_score(pm["old"], pm["score"])})
del tms_tech
tech = rep[["tissue", "cell_type", "compartment", "transfer_auroc_cell", "transfer_auroc_mouse"]].merge(
    pd.DataFrame(tech_rows), on=["tissue", "cell_type"])
tech["expression_minus_technical"] = tech["transfer_auroc_cell"] - tech["tech_transfer_auroc_cell"]
tech.round(3).to_csv(A.figure("replication_kimmel_technical.csv"), index=False)
print(f"technical model ranks Kimmel old above young (cell AUROC > 0.5) in {(tech['tech_transfer_auroc_cell'] > 0.5).sum()} of {len(tech)} rows; "
      f"median {tech['tech_transfer_auroc_cell'].median():.2f}")
print(f"expression beats technical by > 0.05 in {(tech['expression_minus_technical'] > 0.05).sum()} rows, "
      f"technical beats expression by > 0.05 in {(tech['expression_minus_technical'] < -0.05).sum()}")
tech.sort_values("expression_minus_technical").round(2)

### 6. Per-mouse view

Each Kimmel mouse's mean "old" score from the TMS model. Transfer is convincing when every old mouse sits above every young mouse, not only when the cell-level AUROC is high.

In [ ]:
ms = pd.read_csv(A.figure("replication_kimmel_by_mouse.csv"))
ms["row"] = ms["tissue"] + "\n" + ms["cell_type"].str.slice(0, 24)
ms["age"] = np.where(ms["old"] == 1, "old", "young")
g = sns.catplot(data=ms, x="age", y="score", col="row", col_wrap=5, kind="strip", order=["young", "old"],
                height=2.3, size=7, sharey=False)
g.set_titles("{col_name}", size=7)
g.savefig(A.figure("replication_kimmel_by_mouse.png"), dpi=200, bbox_inches="tight")

### 7. Do the key genes replicate, gene by gene?

A whole-transcriptome model can fail to transfer while its key genes still replicate (or the reverse). Per Kimmel mouse: mean expression of the genes the TMS result rests on (*Cdkn1a* = p21, complement *C3* and *Cfb*, collagens), old minus young. `*` = every old mouse above every young one, `v` = every old mouse below.

In [ ]:
genes = ["Cdkn1a", "C3", "Cfb", "Col1a1", "Col3a1", "Mfap5"]
rows = []
for r in match.itertuples():
    k = kim[r.kim_tissue]
    kk = k[(k.obs[r.kim_col] == r.kim_label).to_numpy()]
    g = [x for x in genes if x in k.var_names]
    df = pd.DataFrame(kk[:, g].X.toarray(), columns=g).assign(mouse=kk.obs["mouse"].to_numpy())
    pm = df.groupby("mouse").mean()
    old = pm.index.str.startswith("old")
    out = {"tissue": r.tissue, "cell_type": r.cell_type, "robust": r.robust}
    for x in g:
        d = pm.loc[old, x].mean() - pm.loc[~old, x].mean()
        mark = "*" if pm.loc[old, x].min() > pm.loc[~old, x].max() else ("v" if pm.loc[old, x].max() < pm.loc[~old, x].min() else "")
        out[x] = f"{d:+.2f}{mark}"
    rows.append(out)
genes_kim = pd.DataFrame(rows)
genes_kim.to_csv(A.figure("replication_kimmel_genes.csv"), index=False)
genes_kim

### 8. Is the TMS p21 rise handling stress?

p21 is a stress-response gene, and the dissociation gene *Atf3* also rose in old TMS lung stroma. Per TMS mouse: if p21 only tracks dissociation stress, removing each mouse's dissociation level (mean of `ag.DISSOCIATION` genes) should remove its age trend.

In [ ]:
from scipy.stats import spearmanr
out = []
for t, cts in [("Lung", ["fibroblast of lung", "adventitial cell"]),
               ("Kidney", ["kidney collecting duct principal cell"]), ("Limb_Muscle", ["endothelial cell"])]:
    a = sc.read_h5ad(ag.paths(t).processed)
    D = [g for g in ag.DISSOCIATION if g in a.var_names]
    for ct in cts:
        s = a[(a.obs["cell_ontology_class"] == ct).to_numpy()]
        df = pd.DataFrame({"mouse": s.obs["mouse.id"].astype(str).to_numpy(), "age": s.obs["age"].astype(str).to_numpy(),
                           "p21": s[:, "Cdkn1a"].X.toarray().ravel(), "diss": np.asarray(s[:, D].X.mean(axis=1)).ravel()})
        pm = df.groupby("mouse").agg(age=("age", "first"), n=("p21", "size"), p21=("p21", "mean"), diss=("diss", "mean"))
        pm = pm[pm["n"] >= 10].assign(months=lambda d: d["age"].map(ag.AGE_MONTHS))
        resid = pm["p21"] - np.polyval(np.polyfit(pm["diss"], pm["p21"], 1), pm["diss"])
        rho, p = spearmanr(pm["months"], resid)
        out.append({"tissue": t, "cell_type": ct, "mice": len(pm),
                    "p21_vs_dissociation": spearmanr(pm["p21"], pm["diss"])[0],
                    "age_vs_p21": spearmanr(pm["months"], pm["p21"])[0],
                    "age_vs_p21_without_dissociation": rho, "p": p})
    del a
pd.DataFrame(out).round(3)

### Findings

19 TMS rows matched to Kimmel cell types (lung, kidney, spleen; 6–7 mice per tissue, male, 7–8 vs about 20–23 months). Labels checked by markers: TMS lung fibroblast = Kimmel *Npnt*/*Inmt*/*Tcf21* stromal cells, adventitial = *Dcn*/*Pi16*.

- **A weak age direction transfers across labs.** In all 19 rows the TMS model scores Kimmel's old cells above its young ones (cell AUROC 0.51–0.76), and per-mouse AUROC is 1.00 in 13. The TMS clock places Kimmel's old mice older than its young in all 19 rows, compressed (0.7–5 months against a true gap of about 14). A model trained in one lab can't know the other lab's batch, so this is evidence for a shared, weak age signal.
- **Part of that transfer is technical (5b).** A model given only five technical features per cell (depth, genes per cell, ribosomal, dissociation and hemoglobin fractions; Kimmel has no hemoglobin genes) also ranks Kimmel's old cells above young in 12 of 19 rows (median 0.54). Expression beats it by more than 0.05 in 10 rows: alveolar macrophage (0.72 vs 0.44), kidney fenestrated endothelium (0.64 vs 0.38), collecting duct (0.63 vs 0.37), non-classical monocyte, proximal tubule, kidney macrophage, type II pneumocyte, classical monocyte, spleen B cell and adventitial cell. In the other 9 (lung B, NK, CD4 and CD8 T; spleen NK, T and macrophage; kidney mesangial cell; lung fibroblast) technical features transfer as well or better, so their transfer cannot be told apart from a shared difference in RNA content or handling. A technical difference can itself be biological (old cells may hold less RNA), so this marks those rows as unresolved, not as false. The 19 rows come from 3 tissues of the same 6–7 Kimmel mice.
- **The TMS batch check does not predict which rows transfer.** Cross-age AUROC inside TMS vs transfer AUROC: ρ = 0.19 (p = 0.45). Robust rows median 0.64, others 0.59 (p = 0.22). Removing the sex genes from the verdicts moved one matched row into the robust set (Spleen B cell) and one out (Kidney macrophage, now void by its shuffled control at 0.62); the transfer scores themselves never used sex genes. Lung classical monocytes, only partial in TMS, transfer best (0.76); lung fibroblasts, the strongest TMS row (0.95), transfer worst (0.51; 0.52 without the pooled 3m samples). Read as: the cross-age check is a filter against TMS-specific artifacts, not a predictor of portable signal, and myeloid immune cells (monocytes, macrophages) carry a weak age signal that the TMS design hides and that technical features do not explain; for lymphocytes this is unresolved.
- **Complement replicates, p21 does not, in structural cells.** In Kimmel, *C3* rises in adventitial fibroblasts with every old mouse above every young (+0.61), and in kidney mesangial cells *Cfb* separates fully (+0.43) with *C3* rising too (+0.52). *Cdkn1a* (p21) does not rise in any matched structural row (lung fibroblast −0.03, adventitial −0.46, AT2 −0.21 with every old mouse below every young). In Kimmel's lung **immune** cells it does rise, with full separation: classical monocyte +0.34, CD4 T +0.30, non-classical monocyte +0.20 — the reverse of the TMS pattern.
- **The TMS p21 rise is not handling stress.** Removing each TMS mouse's dissociation level leaves the age trend of p21 in place (lung adventitial ρ 0.86, p = 0.003; fibroblast 0.67, p = 0.017). Since Kimmel's young mice are 7–8 months and TMS has no mice between 3 and 18 months, p21 may rise between 3 and 7 months (late maturation) rather than with aging; neither dataset can tell. **p21 is therefore reported as "rises from 1–3 to 18+ months in TMS, flat from 7 to 21 months elsewhere", not as an aging marker.**